# TODO 6.3：GPTQ INT4 实验

使用与 AWQ 相同的校准集、bit 数和 group size 32/64/128。完整模型写入 `banking_llm_project/gptq`，分析文件写入 `OptimizationSystem/out/gptq`。

In [ ]:
%pip -q install -U gptqmodel optimum accelerate safetensors
# 安装完成后如 Colab 提示重启，请重启运行时再从头运行。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import gc, json, sys
from pathlib import Path
import torch
PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
CONFIG_PATH = PROJECT_DIR / 'configs/gptq.json'
CALIBRATION_MANIFEST = PROJECT_DIR / 'out/dataset_input/dataset_manifest.json'
RESULT_DIR = PROJECT_DIR / 'out/gptq'
MODEL_RESULT_DIR = Path('/content/drive/MyDrive/banking_llm_project/gptq')
sys.path.insert(0, str(PROJECT_DIR))
from src.data.finqa_assets import load_assets
from src.quantization.gptq import build_gptq_plans, run_gptq_plan, collect_gptq_evidence, validate_gptq_evidence
assets = load_assets(DATASET_DIR)
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
plans = build_gptq_plans(config)
assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'
print(torch.cuda.get_device_name(0), [plan.plan_id for plan in plans])

In [ ]:
results = []
for plan in plans:
    results.append(run_gptq_plan(
        plan=plan, model_manifest_file=MODEL_MANIFEST,
        calibration_prompts=assets.calibration_prompts,
        output_dir=RESULT_DIR / plan.plan_id,
        model_output_dir=MODEL_RESULT_DIR / plan.plan_id,
        max_calibration_samples=int(config['max_calibration_samples']),
        max_calibration_seq_len=int(config['max_calibration_seq_len']),
    ))
    gc.collect(); torch.cuda.empty_cache()
print(json.dumps(results, ensure_ascii=False, indent=2))

In [ ]:
evidence = collect_gptq_evidence(CONFIG_PATH, CALIBRATION_MANIFEST, MODEL_MANIFEST, RESULT_DIR)
evidence_path = RESULT_DIR / 'gptq_evidence.json'
evidence_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
report = validate_gptq_evidence(evidence_path)
(RESULT_DIR / 'validation.json').write_text(json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report.to_dict(), ensure_ascii=False, indent=2))
assert report.complete, 'GPTQ 三个 group size 未全部完成。'